In [6]:
# building the position extractors from track file
tracks = "/data/Maelys/visualisation_tools/hicue/tests_data/Scc1.bw"
gff = "/data/Maelys/visualisation_tools/hicue/tests_data/S288c_genes.gtf"
percentage = 5
high_low = "low"

In [7]:
# basic libraires
import numpy as np
import pandas as pd

# system
import importlib
import shutil
import sys
import os

# threading & multiprocessing
from multiprocessing import SimpleQueue
from multiprocessing import Queue as MultiQueue
from multiprocessing import Process
from queue import Queue, Empty
import threading
import asyncio

# biological data
from BCBio import GFF
import pyBigWig

In [3]:
class PositionList():
    position_columns = ["Name", "Chromosome", "Start", "End", "Strand", "Tracks"]
    position_lock = threading.Lock()
    
    def __init__(self, nb_pos, high_low, binning = 1000):
        self._positions = {i:{} for i in range(nb_pos)}
        self._values = [np.inf if high_low == 'low' else -np.inf] * nb_pos
        self._high_low = high_low
        
    @staticmethod 
    def add_value_in_sorted_up(val, val_list, position, position_dict):
        """Adds a value in a ascending sorted list and returns the insertion index, or -1 if not inserted.
        Re-arranges the associated dictionnary."""
        if val <= val_list[0]:
            return -1
        val_list[0] = val
        position_dict[0] = position
        for i in range(len(val_list) - 1):
            if val_list[i] > val_list[i + 1]:
                val_list[i], val_list[i + 1] = val_list[i + 1], val_list[i]
                position_dict[i], position_dict[i + 1] = position_dict[i + 1], position_dict[i]
            else:
                break

    @staticmethod
    def add_value_in_sorted_down(val, val_list, position, position_dict):
        """Adds a value in a ascending sorted list and returns the insertion index, or -1 if not inserted.
        Re-arranges the associated dictionnary."""
        if val >= val_list[0]:
            return -1
        val_list[0] = val
        position_dict[0] = position
        for i in range(len(val_list) - 1):
            if val_list[i] <= val_list[i + 1]:
                val_list[i], val_list[i + 1] = val_list[i + 1], val_list[i]
                position_dict[i], position_dict[i + 1] = position_dict[i + 1], position_dict[i]
            else:
                break
        
    def append(self, position, value):
        """Appends a position to the position list at its value indexation in the values list."""
        with self.position_lock:
            match self._high_low:
                case "high":
                    self.add_value_in_sorted_up(value, self._values, position, self._positions)
                case "low":
                    self.add_value_in_sorted_down(value, self._values, position, self._positions)
                
    def get_positions(self):
        """Returns the position list as a Panda dataframe."""
        return pd.DataFrame.from_dict(self._positions, orient = "index")

In [4]:
# tracks usage

def initialize_globals():
    pass

class TracksPercentageParser(threading.Thread):
    
    def __init__(self, position_list, tracks, input_queue, **kwargs):
        super(TracksPercentageParser, self).__init__(**kwargs)
        
        self._position_list = position_list
        self._tracks = tracks
        self._input_queue = input_queue

        self.start()

    def run(self):
        while True:
            try:
                val = self._input_queue.get()
            except Empty:
                break
            if val == 'DONE':
                break
            index, chrom, start, stop,  = val
            tracks_value = self._tracks.stats(chrom, start - 1, stop - 1)[0]
            position = {
                "Name": f"Locus_{index}",
                "Chromosome": chrom,
                "Start": start,
                "End": stop,
                "Strand": 0,
                "Tracks": tracks_value
            }
            self._position_list.append(position, tracks_value)
            
class TracksThresholdParser(threading.Thread):
    
    def __init__(self, threshold, tracks, input_queue, output_queue, **kwargs):
        super(TracksThresholdParser, self).__init__(**kwargs)
        
        self._min_max, self._threshold = threshold
        self._tracks = tracks
        self._input_queue = input_queue
        self._output_queue = output_queue
        
        self.start()

    def run(self):
        while True:
            try:
                val = self._input_queue.get()
            except Empty:
                break
            if val == 'DONE':
                break
            index, chrom, start, stop = val
            tracks_value = self._tracks.stats(chrom, start - 1, stop - 1)[0]
            to_keep = (tracks_value >= self._threshold  and self._min_max == "min") or (tracks_value <= self._threshold  and self._min_max == "max")
            if to_keep:
                position = {
                    "Name": f"Locus_{index}",
                    "Chromosome": chrom,
                    "Start": start,
                    "End": stop,
                    "Strand": 0,
                    "Tracks": tracks_value
                }
                self._output_queue.put(position)
            
class TracksStreamer(threading.Thread):
    def __init__(self, tracks, output_queue, binning = 1000, threads = 8, **kwargs):
        super(TracksStreamer, self).__init__(**kwargs)
        
        self._tracks = pyBigWig.open(tracks) if type(tracks) == str else tracks
        self._output_queue = output_queue
        self._binning = binning
        
        self._threads = threads
        
        self.start()
        
    def run(self):
        index = 0
        for chrom, size in self._tracks.chroms().items():
            for k in range(0, size, self._binning):
                if k + 1 + self._binning < size:
                    self._output_queue.put((index, chrom, k + 1, k + self._binning))
                    index += 1
            if k + self._binning < size:
                self._output_queue.put((index, chrom, k + 1, size))
        for _ in range(self._threads):
            self._output_queue.put("DONE")

In [109]:
def compute_nb_pos_tracks(tracks, percentage, binning):
    """Computes the number of position that will be kept from the binned tracks to get the right percentage."""
    chromosomes = tracks.chroms()
    nb_pos_tot = 0
    for chrom in chromosomes:
        nb_pos_tot += chromosomes[chrom] // binning + 1
    return round(nb_pos_tot * percentage / 100)

def compute_nb_pos_gff(gff, percentage, gff_types = ["gene"]):
    """Computes the number of position that will be kept from the gff selected type to get the right percentage."""
    examiner = GFF.GFFExaminer()
    in_handle = open(gff)
    gff_type_counts = examiner.available_limits(in_handle)['gff_type']
    nb_pos_tot = 0
    for gff_type, count in gff_type_counts.items():
        nb_pos_tot += count if gff_type[0] in gff_types else 0
    return round(nb_pos_tot * percentage / 100)

binning = 1000
threads = 8
bw = pyBigWig.open(tracks)

input_queue = Queue()
nb_pos = compute_nb_pos_tracks(bw, percentage, binning)
position_list = PositionList(nb_pos, high_low, binning = binning)

workers = [TracksPercentageParser(position_list, bw, input_queue) for i in range(threads)]

streamer = TracksStreamer(tracks, input_queue, binning = binning, threads = threads)
streamer.join()

for worker in workers:
    worker.join()
    
final_list = position_list.get_positions()
print(nb_pos, len(final_list))
print(final_list)

165 165
           Name Chromosome    Start      End  Strand    Tracks
0    Locus_2055         IV  1242001  1243000       0  0.213582
1     Locus_615         II   615001   616000       0  0.213075
2    Locus_1649         IV   836001   837000       0  0.212985
3    Locus_1707         IV   894001   895000       0  0.212950
4    Locus_2640        XVI   296001   297000       0  0.211202
..          ...        ...      ...      ...     ...       ...
160  Locus_3189        XVI   845001   846000       0  0.000000
161  Locus_3192        XVI   848001   849000       0  0.000000
162  Locus_3195        XVI   851001   852000       0  0.000000
163  Locus_3196        XVI   852001   853000       0  0.000000
164  Locus_3289        XVI   945001   946000       0  0.000000

[165 rows x 6 columns]


In [111]:
threshold = 3.0
min_max = "min"
binning = 1000
threads = 8
bw = pyBigWig.open(tracks)

input_queue = Queue()
output_queue = Queue()

workers = [TracksThresholdParser((min_max, threshold), bw, input_queue, output_queue) for i in range(threads)]

streamer = TracksStreamer(tracks, input_queue, binning = binning, threads = threads)
streamer.join()

for worker in workers:
    worker.join()

output_queue.put("DONE")    

positions = []
while True: 
    value = output_queue.get()
    if value != "DONE":
        positions.append(value)
    else:
        break
        
print(pd.DataFrame(positions))

           Name Chromosome   Start     End  Strand    Tracks
0       Locus_9         II    9001   10000       0  4.708780
1      Locus_79         II   79001   80000       0  4.354011
2      Locus_80         II   80001   81000       0  3.783641
3      Locus_93         II   93001   94000       0  4.272153
4     Locus_115         II  115001  116000       0  3.090150
..          ...        ...     ...     ...     ...       ...
176  Locus_3148        XVI  804001  805000       0  5.260985
177  Locus_3179        XVI  835001  836000       0  3.308645
178  Locus_3230        XVI  886001  887000       0  3.270543
179  Locus_3253        XVI  909001  910000       0  3.283171
180  Locus_3279        XVI  935001  936000       0  3.348741

[181 rows x 6 columns]


In [12]:
print(position_list._values)

[0.21358208665559003, 0.21307512527859365, 0.21298525728251752, 0.21294998264199383, 0.2112021842660071, 0.21064281075805039, 0.21063322987448466, 0.21047489104924855, 0.2097317938719903, 0.20950372611080204, 0.20811120579550574, 0.20808132180759498, 0.2076338038325787, 0.20721069231882944, 0.20701201182526272, 0.20425185471683652, 0.2041287237295398, 0.20353438573198634, 0.20212267903057304, 0.19993728815435288, 0.19901048576270974, 0.19843013797167902, 0.1968665883780361, 0.19618380265133278, 0.19566222778460882, 0.19498734019540093, 0.19425409823074355, 0.19415056408458464, 0.1932355003455678, 0.19153444633424818, 0.1898078130440669, 0.1897636564211683, 0.18941363195578256, 0.18754915812531034, 0.18698096840648917, 0.18685228518537572, 0.18367417801070857, 0.1827355130940109, 0.18253279698115807, 0.1823015393609101, 0.17839899222302186, 0.17822039230270786, 0.17476682475692518, 0.17286998822584046, 0.17247972720572063, 0.1723469924983439, 0.17214757367759734, 0.16969877523225588, 0.

In [13]:
values = [] # checking if the computation offers the same results
for chrom, size in bw.chroms().items():
    for k in range(0, size, binning):
        if k + 1 + binning < size:
            values.append(bw.stats(chrom, k , k + binning - 1)[0])
    if k + binning < size:
            values.append(bw.stats(chrom, k , size - 1)[0])
values = np.array(values) 
            
quantile = np.nanquantile(values, (percentage)/100)
print(np.sort(values[values <= quantile]))

[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.00318551 0.00816937 0.01852772 0.02299775 0.02809117 0.03098108
 0.03975916 0.05090681 0.05237779 0.05856118 0.06027948 0.06066221
 0.06281926 0.06350869 0.06535496 0.06535496 0.07179242 0.07570344
 0.08726684 0.08970063 0.09419295 0.10328948 0.10332133 0.10373693
 0.10420481 0.10802683 0.11081387 0.11617983 0.12260822 0.12505879
 0.12548672 0.13

In [14]:
def add_value_in_sorted_up(val,val_list):
    if val <= val_list[0]:
        return -1
    val_list[0] = val
    index = 0
    for i in range(len(val_list) - 1):
        if val_list[i] > val_list[i + 1]:
            val_list[i], val_list[i + 1] = val_list[i + 1], val_list[i]
            index += 1
        else:
            return index
    return index
        
val_list = [1, 2, 5, 10]
print(add_value_in_sorted_up(3, val_list))
print(val_list)
print(add_value_in_sorted_up(6, val_list))
print(val_list)
print(add_value_in_sorted_up(0, val_list))
print(val_list)

1
[2, 3, 5, 10]
2
[3, 5, 6, 10]
-1
[3, 5, 6, 10]


In [15]:
def add_value_in_sorted_down(val, val_list):
    if val >= val_list[0]:
        return -1
    val_list[0] = val
    index = 0
    for i in range(len(val_list) - 1):
        if val_list[i] <= val_list[i + 1]:
            val_list[i], val_list[i + 1] = val_list[i + 1], val_list[i]
            index += 1
        else:
            return index
    return index
        
val_list = [10, 5, 2, 1]
print(add_value_in_sorted_down(3, val_list))
print(val_list)
print(add_value_in_sorted_down(6, val_list))
print(val_list)
print(add_value_in_sorted_down(0, val_list))
print(val_list)

1
[5, 3, 2, 1]
-1
[5, 3, 2, 1]
3
[3, 2, 1, 0]


In [16]:
def compute_nb_pos(tracks, percentage, binning):
    """Computes the number of position that will be kept from the binned tracks to get the right percentage."""
    opened_tracks = pyBigWig.open(tracks)
    chromosomes = opened_tracks.chroms()
    nb_pos_tot = 0
    for chrom in chromosomes:
        nb_pos_tot += chromosomes[chrom] // binning + 1
    print(nb_pos_tot)
    return round(nb_pos_tot * percentage / 100)
    
compute_nb_pos(tracks, percentage, 1000)

3295


165

In [17]:
def compute_nb_pos(gff, percentage, gff_types = ["gene"]):
    """Computes the number of position that will be kept from the gff selected type to get the right percentage."""
    examiner = GFF.GFFExaminer()
    in_handle = open(gff)
    gff_type_counts = examiner.available_limits(in_handle)['gff_type']
    nb_pos_tot = 0
    for gff_type, count in gff_type_counts.items():
        nb_pos_tot += count if gff_type[0] in gff_types else 0
    return round(nb_pos_tot * percentage / 100)
    
compute_nb_pos(gff, percentage)

356

In [114]:
def adjust_locus(locus_position, chromsize, is_circ_chrom = False):
        """Adjust the coordinates of a locus to fit between 0 and the chromosome size. Accounts for circularity."""
        if locus_position < 0:
            locus_position = chromsize + locus_position if is_circ_chrom else 0

        if locus_position >= chromsize:
            locus_position = locus_position - chromsize if is_circ_chrom else chromsize
        return locus_position
    
def bin_tracks(tracks, chrom, start, stop, binning):
    """Returns the binned extracted region from tracks, delimited by start and stop in chrom."""
    values = []
    k = start
    while k < stop:
        end =  k + binning - 1
        end = end if end < stop else stop
        values.append(tracks.stats(chrom, k, end)[0])
        k += binning
    if k < stop:
        values.append(tracks.values(chrom, k, stop)[0])
    return values
    
def extract_tracks(tracks, locus, binning, window, is_loc_circ = False, center="start"): 
    """Extracts a window from tracks around the locus, binned at binning."""
    # 1. computing central coordinate
    match center:
        case "start":
            coordinate = min(locus["Start"], locus["End"]) if locus["Strand"] == 1 or locus["Strand"] == 0 else max(locus["Start"], locus["End"])
        case "center":
            coordinate = (locus["Start"] + locus["End"]) // 2
        case "end":
            coordinate = max(locus["Start"], locus["End"]) if locus["Strand"] == 1 or locus["Strand"] == 0 else min(locus["Start"], locus["End"])

    chrom_size = tracks.chroms(locus["Chromosome"])

    coordinate = adjust_locus(coordinate, chrom_size, is_circ_chrom = is_loc_circ) - 1 # re-ajusting at 0 base for bw format

    # 2. computing binned coordinates
    binned_coordinates = (coordinate // binning)*binning
    start = binned_coordinates
    stop = binned_coordinates + binning - 1
    
    # 3. checking overflows
    is_start_inf = start - window < 0
    is_start_sup = stop + window >= chrom_size
    
    # 3. computing intervales
    window_start = start - window if not is_start_inf else 0
    window_stop = stop + window if not is_start_sup else chrom_size - 1

    # 4. fetching main subtracks
    subtracks = bin_tracks(tracks, locus["Chromosome"], window_start, window_stop, binning)

    # 5. managing overflows
    expected_size = (window//binning) * 2 + 1
    start_overflow = is_start_inf or is_start_sup
    
    if not start_overflow: # no overflow
        return np.array(subtracks)
    
    bins_to_fill = expected_size - len(subtracks)
    fill = [np.nan] * bins_to_fill if not is_loc_circ else []
    
    if is_start_inf and is_loc_circ: # dim 1 inf
        start_inf = (chrom_size//binning - bins_to_fill + 1) * binning
        stop_inf = chrom_size
        fill = bin_tracks(tracks, locus["Chromosome"], start_inf, stop_inf, binning)
        
    if is_start_sup and is_loc_circ: # dim 1 sup
        start_sup = 0
        stop_sup =  bins_to_fill * binning - 1
        fill = bin_tracks(tracks, locus["Chromosome"], start_sup, stop_sup, binning)
        
    to_concat = [fill, subtracks] if is_start_inf else [subtracks, fill]
    subtracks = np.concatenate(to_concat)

    return subtracks

bw = pyBigWig.open(tracks)
window = 200000
for pos in positions:
    print(pos)
    print(extract_tracks(bw, pos, binning, window))

{'Name': 'Locus_9', 'Chromosome': 'II', 'Start': 9001, 'End': 10000, 'Strand': 0, 'Tracks': 4.708779920925488}
[       nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan       

[0.81942163 0.69716178 0.40064453 0.49154643 1.58716593 2.39376443
 0.60000253 2.07387911 2.22198098 0.76722228 0.75353961 2.65250784
 2.34481002 1.04633572 0.43743684 0.58481161 0.4802562  0.51747216
 0.45435992 0.35233827 0.35998477 0.27122812 0.27542335 0.33501668
 0.27873908 0.38226196 0.58574749 2.33534391 2.29108699 0.68277287
 0.32328367 0.27391581 0.28432625 1.03776775 1.13598806 0.36906406
 0.38172557 0.45988354 0.67079568 0.96018752 4.35401132 3.78364086
 0.81084898 0.48171438 0.32388322 0.34122228 0.46830184 0.42651295
 0.33891072 0.43624307 0.23367802 0.42970717 0.54287571 1.15761957
 4.27215316 1.34744736 0.54276319 0.51223508 0.48155998 0.51089754
 0.3336049  0.25186885 0.3269918  0.43343665 0.53732399 1.77497938
 2.29856014 0.51800033 0.25939877 0.4558413  0.71744182 0.5282992
 0.35365501 0.60937574 0.54269206 0.77060281 3.09015041 1.25093906
 0.60209966 1.75899213 3.56450662 0.97214409 0.69025806 0.56060919
 0.38519815 0.39008821 0.67771902 1.30777698 1.63708337 0.76060

[0.90321505 0.22963499 0.40354456 0.75316966 0.66389067 0.34522089
 0.17286999 0.76376121 1.26500263 1.42091122 0.86016494 0.28936793
 0.25745698 0.42015847 0.51837917 1.86573922 4.50367394 1.0534783
 0.84683478 0.97782003 0.66783762 0.66662526 0.57474914 0.4996908
 0.33632059 0.29623226 0.62675197 1.34748161 2.14381385 0.60619935
 0.2490818  0.29409617 0.413944   0.45704053 0.24376117 0.84659916
 0.55864158 0.51848126 0.56515242 1.0383921  7.18093506 2.62586947
 1.07484426 0.79577524 0.81219451 0.5499274  0.39180903 0.26855241
 0.27254903 0.72660214 1.03711664 2.39528317 2.28216904 1.075733
 0.62066818 0.35575001 0.5008265  0.36482361 0.31843339 0.27514107
 0.41906187 1.22891548 2.47715368 3.00361217 1.63592312 1.21185665
 0.78585153 0.51659604 0.23469055 0.28281538 0.63751034 1.24798664
 1.06813687 0.77921192 0.81238088 0.87722649 1.09200398 2.86069147
 6.47005855 1.43207202 0.70660727 0.45499039 0.23699001 0.43965918
 0.83057518 1.23712171 0.60850913 0.23231213 0.22953123 0.01852772

[0.45654896 0.46281356 0.27587229 0.18230154 0.22757749 0.13273216
 0.21700418 0.36093538 0.45660724 0.48540873 0.44893443 0.57163406
 0.6628274  0.44850988 0.59903253 0.94237862 2.9857793  2.75819279
 1.10018174 0.25581512 0.43468581 1.73489463 4.45129961 0.64804478
 0.47639227 0.48903861 0.53837926 0.42025469 0.47812051 0.50368984
 0.56356525 0.41393983 0.36296201 0.33903044 0.22886932 0.23748152
 0.34414998 0.34121731 0.43429813 1.19248438 3.45517094 2.59772233
 0.66380176 1.7065864  1.8033026  0.9724988  0.42132568 0.53445878
 0.56440366 0.41144945 0.40695412 1.27038892 1.38197403 0.64697451
 0.45919062 0.45700133 0.43120303 0.35646457 0.27098213 0.52339873
 1.17296273 2.88855399 0.48176277 0.47504353 0.47440832 0.58245769
 0.54847283 0.79978972 1.10298742 1.99185167 4.35361099 1.02492329
 0.36782974 0.36700922 0.35849213 0.49219984 0.50142971 1.67681028
 0.66093734 0.22139192 0.46470598 0.44222917 0.36711668 0.72603333
 1.48773461 0.55385758 1.11577651 3.38660682 1.19926824 1.2953

[       nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan        nan        nan        nan        nan
        nan        nan 0.37160185 0.05237779 0.60701894 0.06535496
 0.40019663 0.45926821 0.         0.06535496 0.         0.
 0.20811121 0.61187366 0.30665611 0.6253559  1.38827209 0.27548909
 0.71426513 0.77851406 0.63831862 0.33794664 0.37688232 0.45829829
 0.33321694 0.56553821 0.58631366 2.33233561 5.76244747 1.35738374
 1.09265627 0.51361513 0.33627615 0.66212041 0.94686658 2.4786976
 0.7

...

[0.26899878 0.36956602 0.65753647 0.67368303 0.73937144 0.77736909
 0.6232347  0.57696626 0.45940023 0.47865255 0.37556347 0.74572421
 0.57338376 0.43758787 1.50625286 5.31488736 0.93986961 0.5903698
 0.61370677 0.53305793 0.35314339 0.45985899 0.36493583 0.21655242
 0.24813871 0.40412996 1.29534626 0.52797708 0.34950009 0.895379
 0.47566133 0.21928297 0.22687704 0.23656779 0.33260739 0.41060119
 0.37482112 0.37174152 0.5004846  0.50739752 0.65832762 0.88056558
 0.79291827 0.93092861 2.06925646 3.69638373 0.52253215 0.44357112
 0.41387746 0.67569144 1.73008865 3.35191381 0.94590625 0.40814672
 0.51943643 0.36831254 0.51672358 0.22573224 0.37966312 0.41446094
 0.39559298 0.53362815 0.77260696 1.48821487 5.26047353 1.62190265
 0.42277885 0.35250463 0.33660579 0.41329447 0.63461412 0.40170506
 0.64288687 2.948408   1.37596102 0.90651821 2.6621364  0.59981473
 0.54449358 0.99943984 0.40218157 0.37925258 0.36384335 0.24606504
 0.3432586  0.37454086 0.74036115 1.82461662 0.46384525 0.3090688

[0.67657061 0.40362175 0.36029312 0.54443975 0.48982252 0.43644156
 0.28398815 0.21806745 0.30844178 0.29893278 0.54302932 0.3613005
 0.34282131 0.76841061 1.78782961 1.31141598 3.42066571 2.01326935
 0.30583375 0.87784628 0.64354648 0.52697821 0.46607251 1.60590442
 2.73092023 0.79446421 0.48396449 1.03957487 2.36467132 0.59031017
 0.59037944 0.60407834 0.8044266  0.70271118 0.38670812 0.68608482
 2.93871412 1.61681579 0.82278068 0.52907525 0.34217542 0.49371
 1.47698621 2.16571004 0.75357572 0.58140694 0.46348694 0.39688626
 0.45492687 0.36345169 0.27881923 0.32117053 0.33490958 0.89854069
 5.72160969 1.56037762 0.93460643 0.69754264 1.08767449 1.48203744
 1.23911465 0.34393855 0.37936451 0.47119198 0.42370374 0.42979721
 0.50101258 0.5160243  0.46339532 1.11872119 1.04243904 0.33360935
 0.34264325 0.59710365 0.74057562 0.96847494 0.9322286  0.64856095
 0.99209038 4.09874327 3.18083326 0.534384   0.49118951 0.3777018
 0.28585068 0.3087608  1.42197931 3.11208057 0.82252258 0.57690678


[0.38651541 0.41410238 0.73500463 1.55461059 0.58840481 0.50405465
 0.7017673  1.50391079 1.5522212  1.18836681 0.40421301 0.45445545
 0.55706773 0.49482854 0.69764284 0.88583339 3.14217043 4.34702166
 0.46149814 0.39367305 0.30377811 0.36178674 0.31775256 0.57363118
 2.74509723 0.83147011 0.26473525 0.35003487 0.34941981 0.31756318
 0.51583534 0.45356761 0.48915803 0.52489837 0.52165883 1.41335946
 4.4222196  1.72899845 0.92132001 0.61134422 0.52359992 0.54538824
 0.52943309 0.43956349 0.20212268 0.40957425 0.40857986 2.20120139
 0.81178091 0.16969878 0.24201774 0.1825328  0.20721069 0.44009705
 0.51097276 0.70399978 0.58242397 0.69541349 0.5932332  1.18272995
 0.91938982 0.90891841 2.70139221 2.56414073 1.38096135 0.30310466
 0.24324494 0.79187358 1.46034315 0.56144701 0.39434314 0.30375449
 0.21047489 0.30739343 1.08893346 2.61400664 5.2937531  1.9050147
 0.         0.05856118 0.         0.         0.62232162 1.47817509
 1.08790822 1.8016287  0.7145462  0.39376047 0.31204603 0.48804

[0.29400247 0.31939586 0.36089411 0.17247973 0.31304522 0.38208712
 1.17865492 1.41404212 0.40004422 0.81947567 2.49868966 1.09230598
 0.33674383 0.249127   0.21950519 0.35856486 0.3995929  0.34375236
 0.35036031 0.52275099 0.74885405 2.36398546 4.82042709 2.15960072
 0.64833344 0.63109863 0.54221294 0.51028887 0.40591625 0.41497675
 0.6370175  0.75661408 0.35375325 0.78049689 0.79370715 0.89142309
 1.08274544 2.76770864 2.98173364 1.08996781 0.56816377 0.30624038
 0.24866825 0.19993729 0.41945418 0.45140377 0.5559899  0.58209488
 0.8949315  3.65763799 2.12450248 0.46309129 0.1325585  0.44162875
 1.1928481  0.39495868 0.31786347 0.51468137 0.67301437 1.32172309
 3.38458246 3.15027438 2.17660942 0.93851333 0.58645317 0.52287348
 0.48663133 0.37890471 0.5770642  0.6151252  0.40248973 0.60502089
 0.97021933 1.36363842 2.63908758 2.38156682 0.8888256  0.35617762
 0.62064008 0.62238357 0.75794947 1.00157968 1.94905508 8.22840483
 1.64716647 2.59054172 1.67851257 0.95325296 0.67539615 0.7469

[0.71092573 0.49540302 0.43527739 0.43413535 0.32073803 0.40894648
 0.56159385 0.34374958 0.79462409 1.20827079 2.44176809 0.63009923
 0.38155179 1.62696306 3.50035248 0.73175834 0.45352871 0.2258369
 0.26045896 0.23469615 0.34232589 0.58841238 0.41824197 0.43587493
 0.3661347  0.54807048 0.55434303 0.37995493 0.62971132 0.44963957
 0.57036366 0.94829422 1.22812095 2.37436538 3.03257744 0.697792
 0.46574572 0.49797307 2.44347902 1.46780502 0.73692225 0.53726786
 0.3404491  0.40593369 0.64892787 1.10738344 0.44790179 0.55652225
 0.69157769 0.78463951 0.78964437 0.7241052  0.6714837  0.61257861
 1.23117819 1.58680283 1.95849356 0.48327068 1.07977741 0.67166179
 1.88693462 2.60630044 0.52394383 0.55140605 0.6732581  0.48432812
 0.44003799 0.46585076 2.0226075  5.26098494 0.32337486 0.
 0.         0.         0.         0.41708815 0.41679142 0.40993581
 0.48977149 0.60959658 1.92202332 1.17089484 0.64424997 0.69666779
 1.34513566 0.44848993 0.24894162 0.24754232 0.48959886 2.53686779
 1.831

In [105]:
[np.nan] * 10

[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]